<!-- colab-badge -->
[![Abrir no Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bmnogueira-ufms/IA-2026-02/blob/main/Aulas%20Pr%C3%A1ticas/aula02-regressao-linear/aula02-regressao-linear.ipynb)

# Aula Prática 02 - Regressão Linear

**Inteligência Artificial - Graduação**
Prof. Bruno Nogueira - FACOM/UFMS - 2026/2

---

## Do que trata esta aula

Até agora todos os nossos modelos previam **classes discretas**: doente ou saudável,
*setosa* ou *virginica*. Hoje a classe passa a ser um **número real** - o preço de uma
casa, o consumo de um carro - e entramos na tarefa de **regressão**.

E há uma segunda mudança, mais importante: na aula 01 os algoritmos foram
**caixas-pretas**. Hoje não. Vamos **construir a regressão linear do zero**, linha por
linha: a hipótese, a função de custo, o gradiente e o laço de treinamento. No final
comparamos o nosso código com o do scikit-learn - e os números têm que bater.

É a primeira vez no semestre em que você vai **escrever um algoritmo de aprendizado**.

### Roteiro

| # | Tema | Conceito da teoria |
|---|------|--------------------|
| 1 | De classificação para regressão | classe contínua, infinitos valores |
| 2 | A base: consumo de combustível | atributos $x$ e alvo $y$ |
| 3 | A hipótese | $h(x) = \theta_0 + \theta_1 x$ |
| 4 | A função de custo | $J(\theta_0,\theta_1) = \frac{1}{2m}\sum (h(x^{(i)}) - y^{(i)})^2$ |
| 5 | A curva do custo | $J(\theta_1)$ com $\theta_0$ fixo |
| 6 | A superfície do custo | $J(\theta_0,\theta_1)$ em 3D e curvas de nível |
| 7 | O gradiente | derivadas parciais e o sinal da tangente |
| 8 | Gradiente descendente | atualização **simultânea** dos parâmetros |
| 9 | Normalização | *min-max* e escore $z$ |
| 10 | A taxa de aprendizado | $\alpha$ pequeno, bom e grande demais |
| 11 | Conferência | scikit-learn e a solução fechada |
| 12 | Resíduos | o modelo linear é adequado? |
| 13 | Outra base, mesmo código | preço de imóveis |

### ⚠️ Esta aula é para ser preenchida
Este notebook tem **lacunas de propósito**. As células marcadas com 🔨 IMPLEMENTE
contêm apenas a assinatura da função, a documentação e comentários `TODO`; o corpo é
seu. Por isso ele vem **sem saídas**: os gráficos aparecem quando o seu código rodar.

Se travar em alguma delas, o notebook `aula02-gabarito.ipynb` traz a mesma aula com
todas as funções implementadas e comentadas - mas resista à consulta antes de tentar.

### Os dois marcadores desta aula

- 🔨 **IMPLEMENTE** - uma função para você escrever. São **6** ao todo, e elas são o
  coração da aula. Cada uma vem com uma célula de **verificação** logo abaixo: se ela
  imprimir `OK`, sua implementação está certa e você pode seguir.
- 🔎 **PARE E PENSE** - uma pergunta para responder *antes* de rodar a célula seguinte.

Execute as células **na ordem**, de cima para baixo (`Shift + Enter`).

---
## 0. Preparando o ambiente

As mesmas bibliotecas da aula 01. A diferença é que hoje o **NumPy** deixa de ser
detalhe e passa a ser protagonista: toda a matemática da regressão linear vai ser
escrita com operações vetoriais.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

warnings.filterwarnings("ignore")

# Semente aleatória: garante que TODOS na sala obtenham os mesmos números.
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)
plt.rcParams.update({
    "figure.figsize": (7, 4.2),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

# Paleta segura para daltônicos, a mesma da aula 01
CORES = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3"]

print(f"NumPy       {np.__version__}")
print(f"pandas      {pd.__version__}")
print(f"scikit-learn {sklearn.__version__}")
print("\nAmbiente pronto. Hoje nós somos a biblioteca.")

---
## 1. De classificação para regressão

Nos slides: *"alguns problemas requerem modelos preditivos para classes contínuas"*.

A diferença não é de grau, é de natureza. Em classificação o conjunto de respostas é
**contável** - podemos listar as classes e contar acertos. Em regressão há **infinitos**
valores possíveis, e a probabilidade de acertar o valor exato é essencialmente zero.

Isso destrói a acurácia como medida de desempenho. Se um carro faz 12,4 km/l e o
modelo prevê 12,3, ele "errou" - mas errou tão pouco que chamar isso de erro é
absurdo. Em regressão, o que interessa é **o tamanho do erro**, não a sua presença.

O gráfico abaixo mostra as duas situações com o mesmo eixo $x$.

In [ ]:
# Dois problemas, o mesmo atributo no eixo x - só o alvo muda de natureza.
x_demo = np.linspace(0, 10, 40)
y_continuo = 2.5 + 1.4 * x_demo + rng.normal(0, 1.6, x_demo.size)
y_discreto = (x_demo > 5).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

axes[0].scatter(x_demo, y_discreto, c=[CORES[0] if v == 0 else CORES[3] for v in y_discreto],
                s=45, edgecolor="white", zorder=3)
axes[0].set_yticks([0, 1])
axes[0].set_yticklabels(["classe 0", "classe 1"])
axes[0].set_title("CLASSIFICAÇÃO: alvo discreto\n(2 valores possíveis)")
axes[0].set_xlabel("atributo $x$")

axes[1].scatter(x_demo, y_continuo, c=CORES[2], s=45, edgecolor="white", zorder=3)
axes[1].set_title("REGRESSÃO: alvo contínuo\n(infinitos valores possíveis)")
axes[1].set_xlabel("atributo $x$")
axes[1].set_ylabel("alvo $y$")

plt.tight_layout()
plt.show()

print("À esquerda, a pergunta é 'de que lado?'. À direita, é 'quanto?'.")

🔎 **PARE E PENSE**: no gráfico da direita, quantos dos 40 pontos um modelo perfeito
acertaria "na mosca", com todas as casas decimais? E se você arredondar as previsões
para o inteiro mais próximo, a acurácia passa a fazer sentido? O que você perdeu ao
arredondar?

---
## 2. A base: quanto um carro consome?

Os slides usam o exemplo clássico de **tamanho da casa × preço**. Vamos chegar nele -
no fim da aula, com dados reais de imóveis. Mas para *construir* o algoritmo eu quero
uma base pequena, onde cada ponto do gráfico seja visível, e uma pergunta que todo
mundo na sala tem intuição para conferir.

### Auto MPG

**398 automóveis** vendidos nos Estados Unidos entre 1970 e 1982, descritos por peso,
potência, cilindrada, aceleração e ano. O alvo original é o consumo em *miles per
gallon* - vamos converter para **km/l**, e o peso de libras para **quilos**.

- **Fonte original:** UCI Machine Learning Repository - *Auto MPG Data Set*, derivado
  da base do StatLib da Carnegie Mellon University, usada na Exposição da American
  Statistical Association de 1983.
- **Pergunta da aula:** *carro mais pesado gasta mais?* E, sobretudo: **quanto** mais?

In [ ]:
CAMINHO_LOCAL = Path("../../dados/auto-mpg.csv")
URL_FALLBACK = ("https://raw.githubusercontent.com/bmnogueira-ufms/IA-2026-02/"
                "main/dados/auto-mpg.csv")

if CAMINHO_LOCAL.exists():
    bruto = pd.read_csv(CAMINHO_LOCAL)
    print(f"Base carregada do arquivo local: {CAMINHO_LOCAL}")
else:
    bruto = pd.read_csv(URL_FALLBACK)
    print("Base baixada da internet.")

RENOMEAR = {
    "mpg": "consumo_mpg", "cylinders": "cilindros", "displacement": "cilindrada",
    "horsepower": "potencia", "weight": "peso_lb", "acceleration": "aceleracao",
    "model_year": "ano", "origin": "origem", "name": "modelo",
}
carros = bruto.rename(columns=RENOMEAR)

# Conversões para unidades que a gente usa no dia a dia.
carros["peso_kg"] = carros["peso_lb"] * 0.45359237      # 1 lb = 0,45359237 kg
carros["consumo_kml"] = carros["consumo_mpg"] * 0.4251437  # 1 mpg = 0,4251437 km/l

print(f"\n{carros.shape[0]} carros × {carros.shape[1]} colunas")
carros[["modelo", "ano", "peso_kg", "potencia", "consumo_kml"]].head()

In [ ]:
# Antes de modelar, olhe os dados. Sempre.
print(carros[["peso_kg", "potencia", "cilindrada", "consumo_kml"]].describe().round(2))
print("\nValores ausentes por coluna:")
print(carros.isna().sum()[lambda s: s > 0])

Há 6 valores ausentes em `potencia`. Não vamos usar essa coluna na parte principal da
aula, então podemos seguir - mas repare que **verificar isso antes** já é hábito.

Agora o gráfico que vai nos acompanhar por toda a aula: **peso** no eixo $x$,
**consumo** no eixo $y$. Cada ponto é um carro.

In [ ]:
x = carros["peso_kg"].to_numpy(dtype=float)
y = carros["consumo_kml"].to_numpy(dtype=float)
m = x.size

fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.scatter(x, y, s=28, c=CORES[0], alpha=0.65, edgecolor="white", linewidth=0.5, zorder=3)
ax.set_xlabel("peso do carro (kg)")
ax.set_ylabel("consumo (km/l)")
ax.set_title(f"Auto MPG - {m} carros")
plt.tight_layout()
plt.show()

print(f"m = {m} exemplos de treinamento")
print(f"peso:    de {x.min():7.1f} a {x.max():7.1f} kg")
print(f"consumo: de {y.min():7.2f} a {y.max():7.2f} km/l")
print(f"\nCorrelação de Pearson entre peso e consumo: {np.corrcoef(x, y)[0, 1]:.3f}")

🔎 **PARE E PENSE**: a nuvem desce da esquerda para a direita - carro pesado gasta
mais, como esperado. Antes de continuar, **estime de cabeça**: se um carro engordar
100 kg, quantos km/l ele perde? Guarde o seu número. No fim da aula o algoritmo vai
responder isso, e vamos comparar.

---
## 3. A hipótese: uma reta

Nos slides: *"a hipótese na regressão linear é a função linear dos dados que melhor se
adequa aos dados"*. Em duas dimensões, uma **reta**:

$$h(x) = \theta_0 + \theta_1 x$$

Dois parâmetros, e só:

- $\theta_0$ - o **intercepto**: onde a reta cruza o eixo $y$, isto é, a previsão para
  um carro de peso zero (fisicamente sem sentido, matematicamente indispensável);
- $\theta_1$ - a **inclinação**: quanto $y$ muda para cada unidade de $x$. Aqui,
  quantos km/l se ganha (ou perde) por quilo de carro.

**Aprender**, em regressão linear, é escolher esses dois números. Nada mais.

### 🔨 IMPLEMENTE 1 - a hipótese

Escreva a função que, dado um vetor de pesos e os dois parâmetros, devolve o vetor de
previsões. Uma linha basta - e ela deve funcionar para um `x` escalar **ou** para um
vetor NumPy inteiro, sem `for`.

In [ ]:
def hipotese(x, theta0, theta1):
    """Previsão da regressão linear de uma variável.

    Parâmetros
    ----------
    x : float ou np.ndarray
        Valor (ou vetor de valores) do atributo.
    theta0, theta1 : float
        Intercepto e inclinação.

    Retorna
    -------
    float ou np.ndarray
        h(x) = theta0 + theta1 * x, no mesmo formato de x.
    """
    # TODO: devolva theta0 + theta1 * x
    raise NotImplementedError("implemente a hipótese")


# Verificação
assert np.isclose(hipotese(2.0, 1.0, 0.5), 2.0), "h(2) com theta=(1; 0,5) deve dar 2,0"
assert np.allclose(hipotese(np.array([1.0, 2.0, 3.0]), 0.0, 1.0), [1.0, 2.0, 3.0])
assert hipotese(np.zeros(5), 3.0, 7.0).shape == (5,), "deve preservar o formato de x"
print("OK - hipótese implementada.")

Os slides mostram três hipóteses para o mesmo problema: $\theta_1 = 1$, $\theta_1 = 0{,}5$
e $\theta_1 = 0$. Vamos fazer o equivalente com os carros: **três chutes**, escolhidos
a olho, e ver como cada reta se comporta sobre os dados reais.

In [ ]:
chutes = [
    (30.0, -0.010, "chute A"),
    (22.0, -0.003, "chute B"),
    (14.0,  0.000, "chute C (a média, ignorando o peso)"),
]

grade_x = np.linspace(x.min() - 60, x.max() + 60, 200)

fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.scatter(x, y, s=24, c="0.65", alpha=0.6, edgecolor="white", linewidth=0.4, zorder=2)
for (t0, t1, nome), cor in zip(chutes, CORES):
    ax.plot(grade_x, hipotese(grade_x, t0, t1), color=cor, linewidth=2.2, zorder=3,
            label=fr"{nome}: $\theta_0$={t0:g}, $\theta_1$={t1:g}")
ax.set_xlabel("peso do carro (kg)")
ax.set_ylabel("consumo (km/l)")
ax.set_title("Três hipóteses chutadas a olho")
ax.legend(fontsize=8.5, loc="upper right")
plt.tight_layout()
plt.show()

🔎 **PARE E PENSE**: você já consegue *ordenar* as três retas da melhor para a pior só
olhando? Provavelmente sim. Mas agora responda: **quão** melhor é a primeira em
relação à segunda? Duas vezes? Dez por cento?

É exatamente para responder isso que existe a função de custo. Olho humano ordena;
ele não mede.

---
## 4. A função de custo $J$

Nos slides: *"o objetivo é minimizar a diferença entre $h(x)$ e $y$"*. Vamos escrever
isso com precisão. Para o exemplo $i$, o erro é

$$e^{(i)} = h(x^{(i)}) - y^{(i)}$$

e a função de custo dos **mínimos quadrados médios** é

$$J(\theta_0,\theta_1) = \frac{1}{2m}\sum_{i=1}^{m}\left(h(x^{(i)}) - y^{(i)}\right)^2$$

Três decisões escondidas nessa fórmula, e vale entender cada uma:

1. **Por que elevar ao quadrado?** Para que erro para cima e erro para baixo não se
   cancelem, e para punir mais um erro grande do que dois erros pela metade. (Somar
   os valores absolutos também resolveria o cancelamento, mas o quadrado tem derivada
   contínua em todo ponto - o que, como veremos na seção 7, é tudo de que precisamos.)
2. **Por que dividir por $m$?** Para que o custo não dependa do tamanho da base. Sem
   isso, dobrar o número de exemplos dobraria o custo sem que o modelo piorasse.
3. **Por que aquele $2$ no denominador?** Puro conforto: a derivada do quadrado gera
   um fator 2 que cancela com ele. O ponto de mínimo é exatamente o mesmo.

### 🔨 IMPLEMENTE 2 - a função de custo

Escreva $J$. Use operações vetoriais do NumPy: `hipotese(...)` já devolve o vetor de
previsões, então a subtração, o quadrado e a soma são todos elemento a elemento.

> Dica: `np.sum(v ** 2)` ou, equivalente e mais elegante, `v @ v`.

In [ ]:
def custo(x, y, theta0, theta1):
    """Custo dos mínimos quadrados médios (erro quadrático).

    J(theta0, theta1) = (1 / (2m)) * soma_i (h(x_i) - y_i)^2

    Parâmetros
    ----------
    x, y : np.ndarray
        Vetores de mesmo tamanho com o atributo e o alvo.
    theta0, theta1 : float
        Parâmetros da hipótese.

    Retorna
    -------
    float
        O valor do custo.
    """
    m = y.size
    # TODO 1: calcule o vetor de erros (previsão menos valor real)
    # TODO 2: devolva a soma dos erros ao quadrado, dividida por 2m
    raise NotImplementedError("implemente a função de custo")


# Verificação: os números do exemplo dos slides.
# Pontos (1,1), (2,2), (3,3), com theta0 = 0.
x_slide = np.array([1.0, 2.0, 3.0])
y_slide = np.array([1.0, 2.0, 3.0])

for t1, esperado in [(1.0, 0.0), (0.5, 7 / 12), (0.0, 7 / 3)]:
    obtido = custo(x_slide, y_slide, 0.0, t1)
    assert np.isclose(obtido, esperado), f"J(theta1={t1}) deveria ser {esperado:.4f}"
    print(f"J(theta1 = {t1:>3}) = {obtido:.4f}")

print("\nOK - custo implementado, e os três números batem com os slides.")

Vamos **ver** o que a fórmula mede. O gráfico abaixo reproduz os slides: os mesmos
três pontos, as mesmas três hipóteses, e cada erro desenhado como um **segmento
vertical**. O custo é, literalmente, a área média desses segmentos elevada ao quadrado
(dividida por dois).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.9), sharey=True)

for ax, t1 in zip(axes, [1.0, 0.5, 0.0]):
    previsto = hipotese(x_slide, 0.0, t1)
    # Os erros, como segmentos verticais do ponto real até a reta
    for xi, yi, pi in zip(x_slide, y_slide, previsto):
        ax.plot([xi, xi], [yi, pi], color=CORES[3], linewidth=2.4, alpha=0.85, zorder=2)
    ax.plot([0, 3.4], hipotese(np.array([0, 3.4]), 0.0, t1), color=CORES[0],
            linewidth=2.2, zorder=3)
    ax.scatter(x_slide, y_slide, s=70, c=CORES[2], edgecolor="black",
               linewidth=0.8, zorder=4)
    ax.set_title(fr"$\theta_1$ = {t1:g}   →   $J$ = {custo(x_slide, y_slide, 0.0, t1):.3f}")
    ax.set_xlabel("$x$")
    ax.set_xlim(-0.2, 3.4)
    ax.set_ylim(-0.4, 3.6)

axes[0].set_ylabel("$y$")
fig.suptitle("A função de custo mede os segmentos vermelhos (os resíduos)", y=1.03)
plt.tight_layout()
plt.show()

---
## 5. A curva do custo: $J(\theta_1)$

Nos slides há uma passagem sutil e importante. Depois de calcular $J$ para três
valores de $\theta_1$, os três resultados são colocados num **novo gráfico** - e o eixo
horizontal deixa de ser $x$ e passa a ser $\theta_1$.

Isso confunde muita gente, então vale dizer com clareza: **são dois espaços
diferentes**.

| | eixo $x$ | eixo $y$ | cada ponto é |
|---|---|---|---|
| espaço dos **dados** | atributo | alvo | um carro |
| espaço dos **parâmetros** | $\theta_1$ | $J(\theta_1)$ | uma hipótese inteira |

No espaço dos parâmetros, **uma reta do primeiro gráfico virou um único ponto**. E é
nesse segundo espaço que o aprendizado acontece: procuramos o ponto mais baixo.

In [ ]:
grade_t1 = np.linspace(-0.5, 2.5, 300)
custos_t1 = np.array([custo(x_slide, y_slide, 0.0, t) for t in grade_t1])

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))

# Esquerda: espaço dos dados
for t1, cor in zip([1.0, 0.5, 0.0], CORES):
    axes[0].plot([0, 3.4], hipotese(np.array([0, 3.4]), 0.0, t1), color=cor,
                 linewidth=2.0, label=fr"$\theta_1$={t1:g}")
axes[0].scatter(x_slide, y_slide, s=70, c="black", zorder=4)
axes[0].set_xlabel("$x$"); axes[0].set_ylabel("$y$")
axes[0].set_title("espaço dos DADOS\n(cada linha é uma hipótese)")
axes[0].legend(fontsize=9)

# Direita: espaço dos parâmetros
axes[1].plot(grade_t1, custos_t1, color="0.35", linewidth=2.0, zorder=2)
for t1, cor in zip([1.0, 0.5, 0.0], CORES):
    axes[1].scatter([t1], [custo(x_slide, y_slide, 0.0, t1)], s=90, color=cor,
                    edgecolor="black", linewidth=0.8, zorder=4,
                    label=fr"$\theta_1$={t1:g}  $J$={custo(x_slide, y_slide, 0.0, t1):.2f}")
axes[1].axvline(1.0, color=CORES[2], linestyle=":", linewidth=1.4)
axes[1].set_xlabel(r"$\theta_1$"); axes[1].set_ylabel(r"$J(\theta_1)$")
axes[1].set_title("espaço dos PARÂMETROS\n(cada ponto é uma hipótese inteira)")
axes[1].legend(fontsize=8.5)

plt.tight_layout()
plt.show()

print(f"Mínimo da curva: theta1 = {grade_t1[custos_t1.argmin()]:.3f}  "
      f"(J = {custos_t1.min():.5f})")

🔎 **PARE E PENSE**: a curva da direita é uma **parábola** - e isso não é coincidência.
$J$ é uma soma de quadrados de funções lineares em $\theta$; em uma dimensão, isso é
sempre um polinômio de grau 2 com concavidade para cima.

Por que essa forma é uma notícia tão boa para o algoritmo que vamos escrever a seguir?
(Pense em quantos "fundos de vale" existem numa parábola.)

---
## 6. A superfície do custo: $J(\theta_0, \theta_1)$

Com **dois** parâmetros livres, o espaço dos parâmetros é um plano, e $J$ é uma
**superfície** sobre ele - a tigela dos slides. Vamos calculá-la para os dados reais
dos carros e olhar de duas formas: em perspectiva (3D) e de cima (curvas de nível).

> Nota técnica: o peso vale ~1300 kg e o consumo ~10 km/l, então $\theta_1$ vive na
> casa dos **milésimos**. Repare nos números dos eixos - eles vão importar muito na
> seção 9.

In [ ]:
t0_grade = np.linspace(0, 40, 120)
t1_grade = np.linspace(-0.025, 0.005, 120)
T0, T1 = np.meshgrid(t0_grade, t1_grade)

# Cálculo vetorizado da superfície: J para cada par (theta0, theta1) da grade.
# X[None, None, :] cria um eixo extra para as previsões de todos os pares de uma vez.
ERROS = T0[:, :, None] + T1[:, :, None] * x[None, None, :] - y[None, None, :]
J_SUP = (ERROS ** 2).sum(axis=2) / (2 * m)

fig = plt.figure(figsize=(12.5, 4.6))

ax3d = fig.add_subplot(1, 2, 1, projection="3d")
ax3d.plot_surface(T0, T1, J_SUP, cmap="viridis", alpha=0.9, linewidth=0,
                  rstride=2, cstride=2)
ax3d.set_xlabel(r"$\theta_0$"); ax3d.set_ylabel(r"$\theta_1$")
ax3d.set_zlabel(r"$J$")
ax3d.set_title(r"$J(\theta_0,\theta_1)$ em perspectiva")
ax3d.view_init(elev=28, azim=-125)

ax2d = fig.add_subplot(1, 2, 2)
niveis = np.geomspace(J_SUP.min() + 0.5, J_SUP.max(), 25)
cs = ax2d.contour(T0, T1, J_SUP, levels=niveis, cmap="viridis", linewidths=1.0)
i_min = np.unravel_index(J_SUP.argmin(), J_SUP.shape)
ax2d.scatter([T0[i_min]], [T1[i_min]], marker="*", s=220, color=CORES[3],
             edgecolor="black", linewidth=0.6, zorder=5, label="mínimo na grade")
ax2d.set_xlabel(r"$\theta_0$"); ax2d.set_ylabel(r"$\theta_1$")
ax2d.set_title("as mesmas curvas, vistas de cima")
ax2d.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"Melhor par na grade: theta0 = {T0[i_min]:.3f}, theta1 = {T1[i_min]:.6f}, "
      f"J = {J_SUP.min():.4f}")
print(f"Pontos avaliados: {J_SUP.size}")

🔎 **PARE E PENSE**: acabamos de encontrar um bom par de parâmetros por **força
bruta** - avaliamos 14.400 combinações e escolhemos a melhor. Funcionou.

Agora imagine a regressão da próxima aula, com as 8 variáveis da base de imóveis: são
9 parâmetros. Se você quisesse a mesma resolução de 120 valores por parâmetro, quantas
combinações teria de avaliar? Escreva a conta: $120^9$.

Essa é a razão de existir do gradiente descendente. Não é que a força bruta esteja
errada - é que ela morre na primeira base de verdade.

---
## 7. O gradiente: para que lado descer?

A ideia dos slides é minimalista:

> 1. Comece com algum $\theta_0$ e $\theta_1$.
> 2. Modifique-os para reduzir $J$, até chegar a um mínimo.

Estamos com os olhos vendados numa tigela, e queremos chegar ao fundo. A única coisa
que sentimos é a **inclinação do chão sob os pés** - e é exatamente isso que a
derivada informa.

$$\frac{\partial J}{\partial \theta_0} = \frac{1}{m}\sum_{i=1}^{m}\left(h(x^{(i)}) - y^{(i)}\right)
\qquad
\frac{\partial J}{\partial \theta_1} = \frac{1}{m}\sum_{i=1}^{m}\left(h(x^{(i)}) - y^{(i)}\right)x^{(i)}$$

As duas fórmulas saem da regra da cadeia aplicada a $J$ (é a passagem do slide 28).
Note a simetria: as duas são a **média do erro**, e a de $\theta_1$ leva um peso $x^{(i)}$
- faz sentido, porque mexer na inclinação afeta mais os pontos distantes da origem.

E o **sinal** faz todo o trabalho:

| tangente | derivada | $\theta_j - \alpha \cdot$ derivada | efeito |
|---|---|---|---|
| subindo para a direita | positiva | subtrai algo positivo | $\theta_j$ **diminui** (anda para a esquerda) |
| descendo para a direita | negativa | subtrai algo negativo | $\theta_j$ **aumenta** (anda para a direita) |

Nos dois casos, **para o lado do fundo do vale**. Sem saber onde o fundo está.

### 🔨 IMPLEMENTE 3 - o gradiente

Escreva a função que devolve as duas derivadas parciais. Como as duas compartilham o
vetor de erros, calcule-o **uma vez** e reaproveite.

In [ ]:
def gradiente(x, y, theta0, theta1):
    """Derivadas parciais de J em relação a theta0 e theta1.

    Parâmetros
    ----------
    x, y : np.ndarray
        Vetores de mesmo tamanho com o atributo e o alvo.
    theta0, theta1 : float
        Ponto do espaço de parâmetros onde a inclinação é medida.

    Retorna
    -------
    (float, float)
        (dJ/dtheta0, dJ/dtheta1) nessa ordem.
    """
    m = y.size
    # TODO 1: calcule o vetor de erros, como você fez em `custo`
    # TODO 2: d0 é a MÉDIA dos erros
    # TODO 3: d1 é a MÉDIA dos erros PONDERADA por x
    raise NotImplementedError("implemente o gradiente")


# Verificação 1 - a conta do exercício dos slides, feita à mão:
# pontos (0,3), (5,4), (10,5), com theta0 = theta1 = 1.
x_ex = np.array([0.0, 5.0, 10.0])
y_ex = np.array([3.0, 4.0, 5.0])
d0, d1 = gradiente(x_ex, y_ex, 1.0, 1.0)
assert np.isclose(d0, 2.0), "dJ/dtheta0 deveria ser 2,0"
assert np.isclose(d1, 70 / 3), "dJ/dtheta1 deveria ser 70/3 ≈ 23,333"
print(f"Conta dos slides:  dJ/dtheta0 = {d0:.4f}   dJ/dtheta1 = {d1:.4f}")

# Verificação 2 - conferência NUMÉRICA, o teste que pega qualquer erro de álgebra.
# A derivada é o limite de (J(t+eps) - J(t-eps)) / (2*eps). Se a fórmula analítica
# estiver certa, os dois valores coincidem em várias casas decimais.
eps = 1e-6
ponto = (18.0, -0.006)
num0 = (custo(x, y, ponto[0] + eps, ponto[1]) - custo(x, y, ponto[0] - eps, ponto[1])) / (2 * eps)
num1 = (custo(x, y, ponto[0], ponto[1] + eps) - custo(x, y, ponto[0], ponto[1] - eps)) / (2 * eps)
ana0, ana1 = gradiente(x, y, *ponto)
print(f"\nEm theta = {ponto}, sobre os 398 carros:")
print(f"  dJ/dtheta0   analítico {ana0:12.6f}   numérico {num0:12.6f}")
print(f"  dJ/dtheta1   analítico {ana1:12.6f}   numérico {num1:12.6f}")
assert np.isclose(ana0, num0, rtol=1e-4) and np.isclose(ana1, num1, rtol=1e-3)
print("\nOK - gradiente implementado (confere com a derivada numérica).")

Agora o gráfico dos slides 22 a 24: a mesma parábola $J(\theta_1)$, com a **tangente**
desenhada em três pontos. Repare no sinal da inclinação e na direção da flecha.

In [ ]:
# Fixamos theta0 no melhor valor que a busca em grade encontrou na seção 6 e
# fatiamos a superfície ali: sobra uma parábola em theta1, como a dos slides.
t0_fixo = float(T0[i_min])
# theta1 que minimiza J com esse theta0 fixo (o fundo exato desta fatia)
t1_otimo = float(((y - t0_fixo) @ x) / (x @ x))
grade = np.linspace(t1_otimo - 0.011, t1_otimo + 0.011, 300)
curva = np.array([custo(x, y, t0_fixo, t) for t in grade])

fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.plot(grade, curva, color="0.35", linewidth=2.0, zorder=2)

for t1_ponto, cor in zip([t1_otimo - 0.008, t1_otimo, t1_otimo + 0.008],
                         [CORES[0], CORES[2], CORES[3]]):
    j = custo(x, y, t0_fixo, t1_ponto)
    _, incl = gradiente(x, y, t0_fixo, t1_ponto)
    dt = 0.0035
    ax.plot([t1_ponto - dt, t1_ponto + dt], [j - incl * dt, j + incl * dt],
            color=cor, linewidth=2.4, zorder=3)
    ax.scatter([t1_ponto], [j], s=80, color=cor, edgecolor="black", linewidth=0.7, zorder=4)

    if abs(incl) < 1e-6:
        sinal = "ZERO\n(estamos no fundo)"
    else:
        sinal = "positiva" if incl > 0 else "negativa"
        # flecha indicando para onde o gradiente descendente empurra theta1
        ax.annotate("", xy=(t1_ponto - np.sign(incl) * 0.0045, j), xytext=(t1_ponto, j),
                    arrowprops=dict(arrowstyle="-|>", color=cor, linewidth=2.0))
    ax.text(t1_ponto, j + np.ptp(curva) * 0.07, f"derivada = {incl:.1f}\n{sinal}",
            ha="center", fontsize=8.5, color=cor)

ax.set_ylim(top=curva.max() * 1.22)
ax.set_xlabel(r"$\theta_1$")
ax.set_ylabel(fr"$J({t0_fixo:.1f},\ \theta_1)$")
ax.set_title("O sinal da derivada aponta para longe do mínimo -\npor isso o algoritmo SUBTRAI")
plt.tight_layout()
plt.show()

---
## 8. O gradiente descendente

Juntando tudo, o algoritmo dos slides:

> **Repetir até convergir:**
> $$\theta_0 := \theta_0 - \alpha \cdot \frac{1}{m}\sum_{i=1}^{m}\left(h(x^{(i)}) - y^{(i)}\right)$$
> $$\theta_1 := \theta_1 - \alpha \cdot \frac{1}{m}\sum_{i=1}^{m}\left(h(x^{(i)}) - y^{(i)}\right)x^{(i)}$$

E o aviso em letras miúdas, que é a **pegadinha mais comum** da implementação:

> *A atualização de $\theta_0$ e $\theta_1$ deve ser feita ao mesmo tempo.*

Ou seja: as duas derivadas são calculadas com os valores **antigos** dos parâmetros, e
só depois os dois são substituídos. Se você atualizar $\theta_0$ e usar o valor novo
para calcular a derivada de $\theta_1$, o algoritmo passa a descer uma superfície que
não é a sua. Daqui a pouco vamos ver isso acontecer.

### 🔨 IMPLEMENTE 4 - o laço de treinamento

Esta é a função central da aula. Ela deve:

1. partir de `theta0` e `theta1` iniciais;
2. repetir `n_iter` vezes: calcular o gradiente e dar **um passo** de tamanho `alfa`
   na direção oposta, atualizando os dois parâmetros **simultaneamente**;
3. guardar o histórico de $J$ e dos parâmetros a cada iteração - é o histórico que nos
   permite *diagnosticar* o treinamento depois.

> Atenção ao item 2: guarde as duas derivadas em variáveis temporárias antes de
> escrever nos parâmetros. É literalmente o `temp0`/`temp1` do slide 21.

In [ ]:
def gradiente_descendente(x, y, theta0=0.0, theta1=0.0, alfa=0.01, n_iter=1000):
    """Minimiza J por gradiente descendente "batch".

    Parâmetros
    ----------
    x, y : np.ndarray
        Dados de treinamento.
    theta0, theta1 : float
        Ponto de partida.
    alfa : float
        Taxa de aprendizado.
    n_iter : int
        Número de iterações (passos).

    Retorna
    -------
    dict
        {"theta0", "theta1", "historico_J", "historico_theta0", "historico_theta1"}
        Os históricos têm n_iter + 1 valores: o inicial e um por iteração.
    """
    hist_j = [custo(x, y, theta0, theta1)]
    hist_t0 = [theta0]
    hist_t1 = [theta1]

    for _ in range(n_iter):
        # TODO 1: calcule as duas derivadas no ponto ATUAL
        # TODO 2: guarde os novos valores em temp0 e temp1 (ainda sem sobrescrever)
        # TODO 3: só então atribua theta0 = temp0 e theta1 = temp1
        # Substitua a linha abaixo pelo seu código; as três linhas de histórico
        # que vêm depois já estão prontas e não devem ser alteradas.
        raise NotImplementedError("implemente o passo do gradiente descendente")

        hist_j.append(custo(x, y, theta0, theta1))
        hist_t0.append(theta0)
        hist_t1.append(theta1)

    return {
        "theta0": theta0,
        "theta1": theta1,
        "historico_J": np.array(hist_j),
        "historico_theta0": np.array(hist_t0),
        "historico_theta1": np.array(hist_t1),
    }


# Verificação: as três primeiras iterações do exercício dos slides
# (pontos (0,3), (5,4), (10,5); alfa = 0,01; todos os pesos iguais a 1).
r = gradiente_descendente(x_ex, y_ex, theta0=1.0, theta1=1.0, alfa=0.01, n_iter=3)
esperado_t0 = [1.0, 0.98, 0.9718666667, 0.9705702222]
esperado_t1 = [1.0, 0.7666666667, 0.6315555556, 0.5531474074]
assert np.allclose(r["historico_theta0"], esperado_t0), "theta0 não bate com a conta à mão"
assert np.allclose(r["historico_theta1"], esperado_t1), "theta1 não bate com a conta à mão"

print("iter   theta0      theta1          J")
for k in range(4):
    print(f"{k:^4} {r['historico_theta0'][k]:9.6f}  {r['historico_theta1'][k]:9.6f}  "
          f"{r['historico_J'][k]:9.6f}")
print("\nOK - gradiente descendente implementado (bate com a conta à mão dos slides).")

### 8.1 A pegadinha da atualização simultânea

A célula abaixo já vem pronta: é a versão **errada**, que atualiza `theta0` e só depois
calcula a derivada de `theta1` usando o valor já modificado. Compare os dois caminhos.

In [ ]:
def gradiente_descendente_ERRADO(x, y, theta0=0.0, theta1=0.0, alfa=0.01, n_iter=1000):
    """Versão INCORRETA - atualização sequencial em vez de simultânea."""
    hist_j = [custo(x, y, theta0, theta1)]
    for _ in range(n_iter):
        d0, _ = gradiente(x, y, theta0, theta1)
        theta0 = theta0 - alfa * d0          # <-- já mudou aqui...
        _, d1 = gradiente(x, y, theta0, theta1)  # <-- ...e a derivada de theta1 vê o valor NOVO
        theta1 = theta1 - alfa * d1
        hist_j.append(custo(x, y, theta0, theta1))
    return {"theta0": theta0, "theta1": theta1, "historico_J": np.array(hist_j)}


certo = gradiente_descendente(x_ex, y_ex, 1.0, 1.0, alfa=0.01, n_iter=300)
errado = gradiente_descendente_ERRADO(x_ex, y_ex, 1.0, 1.0, alfa=0.01, n_iter=300)

print(f"correto  →  theta0 = {certo['theta0']:.6f}, theta1 = {certo['theta1']:.6f}, "
      f"J = {certo['historico_J'][-1]:.6f}")
print(f"errado   →  theta0 = {errado['theta0']:.6f}, theta1 = {errado['theta1']:.6f}, "
      f"J = {errado['historico_J'][-1]:.6f}")

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(certo["historico_J"], color=CORES[2], linewidth=2, label="atualização simultânea (correta)")
ax.plot(errado["historico_J"], color=CORES[3], linewidth=2, linestyle="--",
        label="atualização sequencial (errada)")
ax.set_xlabel("iteração"); ax.set_ylabel(r"$J$")
ax.set_title("Duas linhas de código trocadas de lugar")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

🔎 **PARE E PENSE**: a versão errada também **desce**, e às vezes desce até um pouco
mais rápido. Esse é justamente o perigo: o bug não estoura, não levanta exceção, não
imprime aviso. Ele só te entrega um resultado ligeiramente diferente do que a teoria
prevê - e você nunca descobre, porque nunca teve o valor "certo" para comparar.

Que estratégia você usaria, na prática, para detectar um bug silencioso desse tipo?
(Você já viu uma delas nesta aula, na seção 7.)

### 8.2 Rodando na base real - e batendo numa parede

Tudo funcionou com números pequenos. Vamos agora treinar nos **398 carros**, com o
mesmo $\alpha = 0{,}01$ do exercício dos slides.

In [ ]:
tentativa = gradiente_descendente(x, y, theta0=0.0, theta1=0.0, alfa=0.01, n_iter=50)

print("iter        J")
for k in [0, 1, 2, 3, 4, 5, 10, 20, 49]:
    print(f"{k:^5} {tentativa['historico_J'][k]:>18.4g}")
print(f"\ntheta0 = {tentativa['theta0']:.4g}   theta1 = {tentativa['theta1']:.4g}")

🔎 **PARE E PENSE**: em vez de diminuir, $J$ explodiu até `inf` (ou `nan`) em poucas
iterações. E não há bug nenhum no seu código - ele acabou de passar em todos os testes.

O que mudou? Olhe a escala. No exercício dos slides, $x$ ia de 0 a 10. Aqui, $x$ é o
peso em quilos: vai a **2300**. A derivada em relação a $\theta_1$ é a média dos erros
**ponderada por $x$** - com $x$ na casa dos milhares, essa derivada é enorme, o passo
$\alpha \cdot d_1$ atropela o mínimo, o erro no outro lado fica maior ainda, e a coisa
diverge em espiral.

Antes de rodar a próxima célula: você consertaria isso mexendo em $\alpha$ ou mexendo
nos **dados**?

In [ ]:
# Primeiro, a tentativa de consertar só com alfa. Funciona?
for a in [1e-2, 1e-4, 1e-6, 1e-7, 1e-8]:
    r = gradiente_descendente(x, y, 0.0, 0.0, alfa=a, n_iter=2000)
    j = r["historico_J"][-1]
    estado = "divergiu" if not np.isfinite(j) else f"J = {j:8.4f}"
    print(f"alfa = {a:<8.0e}  →  {estado:>16}   theta0 = {r['theta0']:8.4f}  "
          f"theta1 = {r['theta1']:+.6f}")

Há uma faixa estreita em que o algoritmo não explode - mas repare no `theta0`: ele fica
praticamente parado em zero. O motivo é geométrico e vale entender bem.

As duas derivadas têm **magnitudes muito diferentes** (uma é a média dos erros, a
outra é essa média multiplicada por ~1300). Com um único $\alpha$ para os dois
parâmetros, o valor que serve para $\theta_1$ é minúsculo para $\theta_0$. A superfície
de $J$, em vez de uma tigela redonda, é um **vale longo e estreito** - e o gradiente
descendente ziguezagueia dentro dele por milhares de iterações.

A solução não é mexer em $\alpha$. É arrumar os dados.

---
## 9. Normalização: pondo os atributos na mesma escala

Os slides tratam isso como *"truque prático"*, e apresentam duas fórmulas:

**Normalização linear** (*min-max*), que joga tudo no intervalo $[0,1]$:

$$l_{ij} = \frac{x_{ij} - \min(x_{\cdot j})}{\max(x_{\cdot j}) - \min(x_{\cdot j})}$$

**Escore $z$** (padronização), que deixa média 0 e desvio padrão 1:

$$z_{ij} = \frac{x_{ij} - \bar{x}_j}{s_j}$$

As duas **preservam a forma da distribuição** - o gráfico de dispersão é o mesmo, só
muda a régua dos eixos. A diferença: *min-max* garante o intervalo mas é sensível a
valores extremos (um único carro absurdo comprime todos os outros); o escore $z$ não
garante intervalo, mas não se deixa dominar por um ponto isolado.

### 🔨 IMPLEMENTE 5 - as duas normalizações

Cada função deve devolver o vetor transformado **e** os dois números usados na
transformação. Guardar esses números é essencial: para prever o consumo de um carro
novo, você tem de aplicar nele **exatamente** a mesma transformação - com a média e o
desvio do **treino**, nunca recalculados.

In [ ]:
def normaliza_minmax(v):
    """Normalização linear para o intervalo [0, 1].

    Retorna
    -------
    (np.ndarray, float, float)
        (vetor normalizado, mínimo usado, máximo usado)
    """
    minimo, maximo = float(v.min()), float(v.max())
    return (v - minimo) / (maximo - minimo), minimo, maximo


def padroniza(v):
    """Normalização por escore z: média 0 e desvio padrão 1.

    Retorna
    -------
    (np.ndarray, float, float)
        (vetor padronizado, média usada, desvio padrão usado)
    """
    # TODO 1: calcule a média e o desvio padrão de v (use float(...) nos dois)
    # TODO 2: devolva (v - média) / desvio, junto com média e desvio
    raise NotImplementedError("implemente a padronização por escore z")


# Verificação
z_teste, mu, sd = padroniza(np.array([2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0]))
assert np.isclose(mu, 5.0) and np.isclose(sd, 2.0), "média 5 e desvio 2 nesse vetor clássico"
assert np.isclose(z_teste.mean(), 0.0) and np.isclose(z_teste.std(), 1.0)
l_teste, lo, hi = normaliza_minmax(np.array([10.0, 20.0, 30.0]))
assert np.allclose(l_teste, [0.0, 0.5, 1.0])
print(f"escore z:  média = {z_teste.mean():.1f}, desvio = {z_teste.std():.1f}")
print("OK - normalizações implementadas.")

In [ ]:
# As três versões do MESMO atributo, lado a lado.
xz, media_x, desvio_x = padroniza(x)
xl, min_x, max_x = normaliza_minmax(x)

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.5))
for ax, (v, titulo, cor) in zip(axes, [
    (x, f"peso original (kg)\nmédia {x.mean():.0f}, desvio {x.std():.0f}", CORES[0]),
    (xl, f"min-max em [0, 1]\nmédia {xl.mean():.2f}, desvio {xl.std():.2f}", CORES[1]),
    (xz, f"escore z\nmédia {xz.mean():.2f}, desvio {xz.std():.2f}", CORES[2]),
]):
    ax.scatter(v, y, s=18, c=cor, alpha=0.6, edgecolor="white", linewidth=0.3)
    ax.set_title(titulo, fontsize=9.5)
    ax.set_xlabel("atributo")
axes[0].set_ylabel("consumo (km/l)")
fig.suptitle("A nuvem é a mesma; só a régua do eixo x mudou", y=1.04)
plt.tight_layout()
plt.show()

### 9.1 Treinando de novo, agora com o atributo padronizado

Mesmo código, mesmo $\alpha$ inicial. A única diferença é o `xz` no lugar do `x`.

In [ ]:
modelo_z = gradiente_descendente(xz, y, theta0=0.0, theta1=0.0, alfa=0.1, n_iter=400)

print(f"theta0 = {modelo_z['theta0']:.4f}   theta1 = {modelo_z['theta1']:.4f}")
print(f"J inicial = {modelo_z['historico_J'][0]:.4f}   "
      f"J final = {modelo_z['historico_J'][-1]:.4f}")
print(f"\nRedução do custo: {100 * (1 - modelo_z['historico_J'][-1] / modelo_z['historico_J'][0]):.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

# Curva de convergência - o gráfico de diagnóstico dos slides
axes[0].plot(modelo_z["historico_J"], color=CORES[0], linewidth=2)
axes[0].set_xlabel("iteração"); axes[0].set_ylabel(r"$J(\theta)$")
axes[0].set_title(r"$J$ × iterações: desce a cada passo, e estabiliza")

# Trajetória sobre as curvas de nível - o caminho até o fundo do vale
t0g = np.linspace(-2, 28, 140)
t1g = np.linspace(-9, 3, 140)
G0, G1 = np.meshgrid(t0g, t1g)
E = G0[:, :, None] + G1[:, :, None] * xz[None, None, :] - y[None, None, :]
JZ = (E ** 2).sum(axis=2) / (2 * m)
axes[1].contour(G0, G1, JZ, levels=np.geomspace(JZ.min() + 0.3, JZ.max(), 22),
                cmap="viridis", linewidths=0.9)
axes[1].plot(modelo_z["historico_theta0"], modelo_z["historico_theta1"],
             color=CORES[3], linewidth=1.6, marker="o", markersize=2.6, zorder=4)
axes[1].scatter([modelo_z["theta0"]], [modelo_z["theta1"]], marker="*", s=240,
                color=CORES[1], edgecolor="black", linewidth=0.6, zorder=5)
axes[1].set_xlabel(r"$\theta_0$"); axes[1].set_ylabel(r"$\theta_1$")
axes[1].set_title("o caminho percorrido no espaço de parâmetros")

plt.tight_layout()
plt.show()

Os slides pedem exatamente esse gráfico da esquerda: *"como saber se está convergindo?
Plotar $J(\theta)$ × número de iterações; $J$ deve diminuir depois de cada iteração"*.
Este é **o** gráfico de diagnóstico da regressão linear.

E o da direita mostra a mesma história no espaço de parâmetros: passos largos no
começo (onde a superfície é inclinada) e cada vez menores perto do fundo, porque o
próprio gradiente encolhe. O algoritmo **desacelera sozinho** - não precisamos reduzir
$\alpha$ manualmente.

Repare também na forma das curvas de nível: agora são **círculos**, não as elipses
esticadas da seção 6. Foi a padronização que arredondou a tigela, e é por isso que o
caminho até o fundo virou praticamente uma linha reta. Guarde essa imagem - ela volta
na seção 10.

### 9.2 Vendo a reta nascer

O histórico guarda os parâmetros de todas as 400 iterações. Vamos usá-lo para desenhar
a hipótese em quatro momentos do treinamento.

In [ ]:
momentos = [0, 3, 15, 400]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.4), sharey=True)
gx = np.linspace(xz.min(), xz.max(), 100)

for ax, k in zip(axes, momentos):
    t0k = modelo_z["historico_theta0"][k]
    t1k = modelo_z["historico_theta1"][k]
    ax.scatter(xz, y, s=14, c="0.7", alpha=0.6, edgecolor="none")
    ax.plot(gx, hipotese(gx, t0k, t1k), color=CORES[3], linewidth=2.4)
    ax.set_title(f"iteração {k}\n$J$ = {modelo_z['historico_J'][k]:.3f}", fontsize=9.5)
    ax.set_xlabel("peso (escore z)")
axes[0].set_ylabel("consumo (km/l)")
axes[0].set_ylim(y.min() - 2, y.max() + 2)
fig.suptitle("A hipótese sendo aprendida, passo a passo", y=1.06)
plt.tight_layout()
plt.show()

### 🔨 IMPLEMENTE 6 - voltando às unidades originais

Temos $\theta$ no mundo padronizado, mas ninguém entende "consumo cai 6 km/l por
desvio padrão de peso". Queremos km/l **por quilo**.

Faça a álgebra. Se $z = \dfrac{x - \bar{x}}{s}$ e $h = a_0 + a_1 z$, então

$$h = a_0 + a_1\frac{x - \bar{x}}{s}
     = \underbrace{\left(a_0 - \frac{a_1\bar{x}}{s}\right)}_{\theta_0}
     + \underbrace{\frac{a_1}{s}}_{\theta_1}\, x$$

Implemente essa conversão.

In [ ]:
def desnormaliza(a0, a1, media, desvio):
    """Converte parâmetros do espaço padronizado para as unidades originais.

    Parâmetros
    ----------
    a0, a1 : float
        Parâmetros aprendidos com o atributo padronizado.
    media, desvio : float
        Os mesmos valores devolvidos por `padroniza`.

    Retorna
    -------
    (float, float)
        (theta0, theta1) na escala original de x.
    """
    # TODO: aplique as duas fórmulas do texto acima
    raise NotImplementedError("implemente a desnormalização")


# Verificação: a previsão tem de ser IDÊNTICA nos dois espaços.
th0, th1 = desnormaliza(modelo_z["theta0"], modelo_z["theta1"], media_x, desvio_x)
assert np.allclose(hipotese(xz, modelo_z["theta0"], modelo_z["theta1"]),
                   hipotese(x, th0, th1)), "as previsões deveriam coincidir"
assert np.isclose(custo(x, y, th0, th1), custo(xz, y, modelo_z["theta0"], modelo_z["theta1"]))
print(f"No espaço padronizado:  h(z) = {modelo_z['theta0']:.4f} + ({modelo_z['theta1']:.4f}) · z")
print(f"Nas unidades originais: h(x) = {th0:.4f} + ({th1:.6f}) · x")
print("OK - desnormalização implementada (as previsões coincidem).")

### 9.3 A resposta da aula

Agora podemos ler o modelo em português.

In [ ]:
fig, ax = plt.subplots(figsize=(7.8, 4.8))
ax.scatter(x, y, s=28, c=CORES[0], alpha=0.55, edgecolor="white", linewidth=0.5, zorder=3)
ax.plot(grade_x, hipotese(grade_x, th0, th1), color=CORES[3], linewidth=2.6, zorder=4,
        label=fr"$h(x) = {th0:.2f} {th1:+.5f}\,x$")
ax.set_xlabel("peso do carro (kg)")
ax.set_ylabel("consumo (km/l)")
ax.set_title("A reta que o nosso algoritmo aprendeu")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"theta0 = {th0:8.4f} km/l   →  previsão para um carro de peso 0 (extrapolação sem sentido físico)")
print(f"theta1 = {th1:8.6f} km/l por kg")
print(f"\nOu seja: a cada 100 kg a MAIS, o consumo cai {abs(th1) * 100:.2f} km/l.")
print(f"Um carro de 1000 kg: {hipotese(1000.0, th0, th1):.2f} km/l")
print(f"Um carro de 1500 kg: {hipotese(1500.0, th0, th1):.2f} km/l")
print(f"Um carro de 2000 kg: {hipotese(2000.0, th0, th1):.2f} km/l")

🔎 **PARE E PENSE**: compare com a estimativa que você guardou na seção 2. Acertou a
ordem de grandeza?

E agora a pergunta perigosa: o modelo diz que **tirar** 100 kg de um carro faz ele
render ~0,5 km/l a mais. Isso é uma afirmação **causal** - e a regressão linear não
autoriza esse salto. Carros pesados também tendem a ter motores maiores, mais
cilindros, câmbio diferente. O que o modelo mediu foi **associação**, não causa. Que
tipo de dado seria necessário para sustentar a afirmação causal?

---
## 10. A taxa de aprendizado $\alpha$

Os slides resumem: $\alpha$ pequeno demais faz o algoritmo demorar a convergir;
$\alpha$ grande demais pode fazê-lo **não convergir**. E a receita prática: *"testar
diferentes valores: 0,001; 0,01; 0,1; 1; ..."*.

Vamos testar essa escala inteira, no atributo padronizado.

In [ ]:
valores_alfa = [0.001, 0.01, 0.1, 0.5, 1.0, 1.9, 2.1]
resultados = {a: gradiente_descendente(xz, y, 0.0, 0.0, alfa=a, n_iter=120)
              for a in valores_alfa}

fig, axes = plt.subplots(1, 3, figsize=(14, 4.0))

# alfa é uma grandeza ordenada, então aqui uma paleta sequencial comunica melhor
# que cores categóricas: quanto mais claro, maior o alfa.
cores_alfa = plt.cm.viridis(np.linspace(0.05, 0.9, len(valores_alfa)))

for a, cor in zip(valores_alfa, cores_alfa):
    hj = resultados[a]["historico_J"]
    axes[0].plot(hj, color=cor, linewidth=1.9, label=fr"$\alpha$ = {a}")
    if np.all(np.isfinite(hj)) and hj[-1] < hj[0]:
        axes[1].plot(hj, color=cor, linewidth=1.9, label=fr"$\alpha$ = {a}")
        axes[2].plot(resultados[a]["historico_theta1"], color=cor, linewidth=1.6,
                     label=fr"$\alpha$ = {a}")

axes[0].set_yscale("log")
axes[0].set_xlabel("iteração"); axes[0].set_ylabel(r"$J$ (escala log)")
axes[0].set_title("todos os valores testados")
axes[0].legend(fontsize=8, ncol=2)

axes[1].set_xlim(0, 60); axes[1].set_ylim(0, 40)
axes[1].set_xlabel("iteração"); axes[1].set_ylabel(r"$J$")
axes[1].set_title("zoom nos que não divergem")
axes[1].legend(fontsize=8, ncol=2)

axes[2].axhline(modelo_z["theta1"], color="0.4", linestyle=":", linewidth=1.4)
axes[2].set_xlim(0, 30)
axes[2].set_xlabel("iteração"); axes[2].set_ylabel(r"$\theta_1$")
axes[2].set_title(r"o caminho de $\theta_1$ (a linha pontilhada é o ótimo)")
axes[2].legend(fontsize=8, ncol=2)

plt.tight_layout()
plt.show()

print(f"{'alfa':>8}  {'J após 120 iterações':>22}  diagnóstico")
for a, r in resultados.items():
    hj = r["historico_J"]
    if not np.all(np.isfinite(hj)) or hj[-1] > hj[0]:
        diag = "DIVERGIU"
    elif hj[-1] > hj[0] * 0.5:
        diag = "convergindo, mas devagar demais"
    elif abs(hj[-1] - hj[-2]) < 1e-9:
        diag = "convergiu"
    else:
        diag = "quase lá"
    print(f"{a:>8}  {hj[-1]:>22.6g}  {diag}")

Três coisas para observar, e cada uma vale um comentário.

**$\alpha = 2{,}1$ diverge; $\alpha = 1{,}9$ não.** A fronteira não é 1 nem 10: para
esta superfície ela fica logo abaixo de 2. Existe, portanto, um $\alpha$ máximo que
depende da **forma da superfície**, não do gosto de quem programa. Em regressão linear
ele é calculável; em redes neurais não é - e é por isso que ajustar $\alpha$ continua
sendo mais arte que ciência.

**Com $\alpha = 1{,}9$, olhe o terceiro gráfico.** O custo cai, mas $\theta_1$ salta de
um lado do mínimo para o outro - $-5{,}2$, $-0{,}5$, $-4{,}8$, $-0{,}9$... - e vai
fechando o cerco devagar. É o ziguezague de que os slides falam, e ele custa iterações.

**Com $\alpha = 1$, o algoritmo acerta em UMA iteração.** Não é sorte: como
padronizamos o atributo, a superfície de $J$ é uma tigela perfeitamente circular, e um
passo de tamanho 1 na direção do gradiente cai exatamente no fundo. Foi a normalização
que produziu essa geometria - mais um argumento a favor dela.

🔎 **PARE E PENSE**: se você só pudesse olhar **um** gráfico para decidir se o seu
$\alpha$ está bom, qual dos três seria? E o que exatamente você procuraria nele?

---
## 11. Conferência: será que acertamos?

Escrevemos ~15 linhas de matemática. Antes de confiar nelas, vamos comparar com duas
referências independentes.

### 11.1 A solução fechada

Para uma variável, o mínimo de $J$ pode ser obtido **sem nenhuma iteração**: basta
igualar as duas derivadas a zero e resolver o sistema. O resultado é

$$\theta_1 = \frac{\sum (x^{(i)} - \bar{x})(y^{(i)} - \bar{y})}{\sum (x^{(i)} - \bar{x})^2}
\qquad \theta_0 = \bar{y} - \theta_1 \bar{x}$$

### 11.2 O scikit-learn

`LinearRegression` resolve o mesmo problema por álgebra linear (decomposição QR),
também sem iterar.

Três caminhos completamente diferentes. Os números têm de bater.

In [ ]:
from sklearn.linear_model import LinearRegression

# Solução fechada
t1_fechada = ((x - x.mean()) @ (y - y.mean())) / ((x - x.mean()) @ (x - x.mean()))
t0_fechada = y.mean() - t1_fechada * x.mean()

# scikit-learn (espera X em duas dimensões: uma linha por exemplo, uma coluna por atributo)
sk = LinearRegression().fit(x.reshape(-1, 1), y)

comparacao = pd.DataFrame({
    "theta0": [th0, t0_fechada, sk.intercept_],
    "theta1": [th1, t1_fechada, sk.coef_[0]],
    "J": [custo(x, y, th0, th1),
          custo(x, y, t0_fechada, t1_fechada),
          custo(x, y, sk.intercept_, sk.coef_[0])],
}, index=["nosso gradiente descendente", "solução fechada", "scikit-learn"])

print(comparacao.to_string(float_format=lambda v: f"{v:.6f}"))
assert np.isclose(t1_fechada, sk.coef_[0]) and np.isclose(t0_fechada, sk.intercept_)
print("\nSolução fechada e scikit-learn são idênticas (mesmo problema, mesma resposta exata).")
print(f"Diferença do nosso theta1 em relação ao exato: "
      f"{abs(th1 - t1_fechada) / abs(t1_fechada) * 100:.4f}%")

🔎 **PARE E PENSE**: se existe fórmula fechada, por que estudamos gradiente
descendente?

Três razões. **(1)** A fórmula fechada da versão multivariada envolve inverter uma
matriz $(n+1) \times (n+1)$ - com muitos atributos isso fica caro e numericamente
instável. **(2)** Ela só existe porque $J$ é quadrática; troque o modelo (regressão
logística, redes neurais) e não há fórmula nenhuma. **(3)** O gradiente descendente
funciona por lotes, então roda em bases que não cabem na memória.

O gradiente descendente é o algoritmo geral. A fórmula fechada é o caso de sorte.

### 11.3 Quanto erra, em unidades do mundo?

$J$ é ótimo para *otimizar*, mas ruim para *relatar*: está em (km/l)² e dividido por 2.
Para comunicar o desempenho usamos as métricas que você já viu na aula 01.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

previsto = hipotese(x, th0, th1)
media_sempre = np.full_like(y, y.mean())   # a linha de base: prever sempre a média

for nome, pred in [("nosso modelo", previsto), ("baseline (prever a média)", media_sempre)]:
    print(f"{nome:>26}   MAE {mean_absolute_error(y, pred):5.3f} km/l   "
          f"RMSE {np.sqrt(mean_squared_error(y, pred)):5.3f} km/l   "
          f"R² {r2_score(y, pred):6.3f}")

print(f"\nO desvio padrão do consumo é {y.std():.3f} km/l - é esse o erro que o baseline comete.")
print("R² é a fração da variância do alvo que o modelo explica: 0 = igual ao baseline, 1 = perfeito.")

---
## 12. Olhando os resíduos: o modelo linear serve aqui?

Um $R^2$ de ~0,7 parece bom. Mas *ajuste bom em média* não quer dizer *modelo
adequado*. O diagnóstico honesto é olhar os **resíduos** $y - h(x)$: se o modelo
capturou tudo que havia de sistemático, o que sobra deve parecer **ruído sem padrão**.

In [ ]:
residuos = y - previsto

fig, axes = plt.subplots(1, 3, figsize=(13, 3.7))

axes[0].scatter(previsto, residuos, s=22, c=CORES[0], alpha=0.6, edgecolor="none")
axes[0].axhline(0, color=CORES[3], linewidth=1.6)
axes[0].set_xlabel("valor previsto (km/l)"); axes[0].set_ylabel("resíduo (km/l)")
axes[0].set_title("resíduo × previsto")

axes[1].scatter(x, residuos, s=22, c=CORES[2], alpha=0.6, edgecolor="none")
axes[1].axhline(0, color=CORES[3], linewidth=1.6)
axes[1].set_xlabel("peso (kg)"); axes[1].set_ylabel("resíduo (km/l)")
axes[1].set_title("resíduo × atributo")

axes[2].hist(residuos, bins=30, color=CORES[4], edgecolor="white")
axes[2].axvline(0, color=CORES[3], linewidth=1.6)
axes[2].set_xlabel("resíduo (km/l)"); axes[2].set_ylabel("carros")
axes[2].set_title("distribuição dos resíduos")

plt.tight_layout()
plt.show()

print(f"média dos resíduos: {residuos.mean():.2e}  (tem de ser ~0: é consequência da derivada em theta0)")
print(f"resíduo mínimo {residuos.min():+.2f} km/l   |   máximo {residuos.max():+.2f} km/l")

🔎 **PARE E PENSE**: o primeiro gráfico tem uma **curva** clara, em forma de U - os
resíduos são positivos nas pontas e negativos no meio. Isso é um padrão, e padrão em
resíduo significa **estrutura que o modelo não capturou**.

A relação entre peso e km/l não é uma reta: é uma curva. Faz sentido? Sim, e por um
motivo físico bonito. O consumo de combustível é aproximadamente proporcional ao
peso, mas km/l é o **inverso** do consumo. Se `litros/km` cresce linearmente com o
peso, então `km/l` decresce como $1/\text{peso}$ - uma hipérbole, não uma reta.

Vamos testar essa hipótese trocando a unidade do alvo para **litros por 100 km** (a
unidade usada na Europa justamente por ser proporcional ao gasto).

In [ ]:
y_l100 = 235.214583 / carros["consumo_mpg"].to_numpy(dtype=float)  # mpg -> L/100 km

fig, axes = plt.subplots(2, 2, figsize=(11.5, 7))

for j, (alvo, rotulo) in enumerate([(y, "km/l"), (y_l100, "litros / 100 km")]):
    mod = gradiente_descendente(xz, alvo, 0.0, 0.0, alfa=0.1, n_iter=400)
    a0, a1 = desnormaliza(mod["theta0"], mod["theta1"], media_x, desvio_x)
    pred = hipotese(x, a0, a1)
    r2 = r2_score(alvo, pred)

    axes[0, j].scatter(x, alvo, s=20, c=CORES[j], alpha=0.55, edgecolor="none")
    axes[0, j].plot(grade_x, hipotese(grade_x, a0, a1), color=CORES[3], linewidth=2.2)
    axes[0, j].set_title(f"alvo em {rotulo}   -   $R^2$ = {r2:.3f}")
    axes[0, j].set_xlabel("peso (kg)"); axes[0, j].set_ylabel(rotulo)

    axes[1, j].scatter(pred, alvo - pred, s=20, c=CORES[j], alpha=0.55, edgecolor="none")
    axes[1, j].axhline(0, color=CORES[3], linewidth=1.5)
    axes[1, j].set_title("resíduos")
    axes[1, j].set_xlabel("previsto"); axes[1, j].set_ylabel("resíduo")
    print(f"alvo em {rotulo:>16}:  R² = {r2:.4f}")

plt.tight_layout()
plt.show()

O $R^2$ sobe de ~0,69 para ~0,78, e a curvatura dos resíduos praticamente desaparece.
**Não mudamos o algoritmo, mudamos a pergunta** - e a mesma reta passou a servir.

Guarde essa lição: quando o modelo linear vai mal, antes de partir para algo mais
complicado, verifique se o problema não está na forma como as variáveis foram
escritas. Os slides chamam isso de escolher um espaço de hipóteses adequado; na
próxima aula vamos fazer isso de propósito, com termos polinomiais.

---
## 13. Outra base, o mesmo código: o exemplo dos slides

Fechamos o círculo com o exemplo que abre a aula teórica: **tamanho da casa × preço**.

### Ames Housing

**2.930 casas** vendidas em Ames, Iowa, entre 2006 e 2010, com 80 atributos por
imóvel. É a base que substituiu a antiga *Boston Housing* no ensino de regressão.

- **Referência:** DE COCK, D. Ames, Iowa: alternative to the Boston housing data as an
  end of semester regression project. *Journal of Statistics Education*, v. 19, n. 3, 2011.
- Vamos usar duas colunas: `Gr Liv Area` (área construída acima do solo, convertida
  para m²) e `SalePrice` (preço de venda, em milhares de dólares).

O código abaixo é **exatamente** o mesmo de antes. Só os dados mudam.

In [ ]:
CAMINHO_AMES = Path("../../dados/ames-housing.csv")
URL_AMES = ("https://raw.githubusercontent.com/bmnogueira-ufms/IA-2026-02/"
            "main/dados/ames-housing.csv")

ames = pd.read_csv(CAMINHO_AMES if CAMINHO_AMES.exists() else URL_AMES)
ames["area_m2"] = ames["Gr Liv Area"] * 0.09290304   # 1 pé² = 0,09290304 m²
ames["preco_mil"] = ames["SalePrice"] / 1000

xa = ames["area_m2"].to_numpy(dtype=float)
ya = ames["preco_mil"].to_numpy(dtype=float)

print(f"{xa.size} casas")
print(f"área:  de {xa.min():6.1f} a {xa.max():6.1f} m²   (média {xa.mean():.1f})")
print(f"preço: de {ya.min():6.1f} a {ya.max():6.1f} mil dólares   (média {ya.mean():.1f})")

# Mesmíssimo pipeline: padroniza, treina, desnormaliza.
xa_z, media_a, desvio_a = padroniza(xa)
mod_a = gradiente_descendente(xa_z, ya, 0.0, 0.0, alfa=0.1, n_iter=600)
ta0, ta1 = desnormaliza(mod_a["theta0"], mod_a["theta1"], media_a, desvio_a)

sk_a = LinearRegression().fit(xa.reshape(-1, 1), ya)
print(f"\nnosso    : theta0 = {ta0:8.3f}   theta1 = {ta1:7.4f}")
print(f"sklearn  : theta0 = {sk_a.intercept_:8.3f}   theta1 = {sk_a.coef_[0]:7.4f}")
print(f"\nR² = {r2_score(ya, hipotese(xa, ta0, ta1)):.3f}")
print(f"Cada m² a mais vale, em média, {ta1 * 1000:.0f} dólares.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))

ga = np.linspace(xa.min(), xa.max(), 100)
axes[0].scatter(xa, ya, s=14, c=CORES[0], alpha=0.4, edgecolor="none")
axes[0].plot(ga, hipotese(ga, ta0, ta1), color=CORES[3], linewidth=2.6,
             label=fr"$h(x) = {ta0:.1f} + {ta1:.3f}\,x$")
axes[0].set_xlabel("área construída (m²)")
axes[0].set_ylabel("preço (mil dólares)")
axes[0].set_title("Ames Housing - o exemplo dos slides, com dados reais")
axes[0].legend(fontsize=9)

axes[1].plot(mod_a["historico_J"], color=CORES[2], linewidth=2)
axes[1].set_yscale("log")
axes[1].set_xlabel("iteração"); axes[1].set_ylabel(r"$J$ (escala log)")
axes[1].set_title("convergência: o mesmo laço, outra base")

plt.tight_layout()
plt.show()

🔎 **PARE E PENSE**: a nuvem é bem mais espalhada que a dos carros, e o $R^2$ fica em
torno de 0,50 - metade da variação do preço fica sem explicação. Nada de errado com o
algoritmo: **a área simplesmente não determina o preço de uma casa**. Bairro, ano de
construção, estado de conservação, número de banheiros - tudo isso está fora do modelo.

Olhe a base: `ames` tem outras 16 colunas ali paradas. Como você usaria mais de uma?
A resposta é a **regressão linear multivariada**, e é onde a próxima aula começa:

$$h(x) = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \dots + \theta_n x_n = \theta^T x$$

O melhor é que quase nada do que você escreveu hoje precisa mudar. O gradiente
vira um vetor, o laço é o mesmo, e a normalização - que hoje foi conveniência - passa
a ser obrigatória.

---
## 14. Fechamento

Você implementou um algoritmo de aprendizado de máquina do zero. Recapitulando o que
cada peça faz:

| peça | função | o que ela responde |
|---|---|---|
| hipótese $h(x)$ | `hipotese` | qual é a previsão para este exemplo? |
| custo $J(\theta)$ | `custo` | quão ruim é esta hipótese? |
| gradiente $\nabla J$ | `gradiente` | para que lado eu ando para melhorar? |
| laço de treino | `gradiente_descendente` | ande até não melhorar mais |
| normalização | `padroniza` | ponha os atributos em escalas comparáveis |

### As quatro ideias que sobrevivem ao algoritmo

1. **Aprender é otimizar.** Definimos uma medida de erro e procuramos os parâmetros
   que a minimizam. Isso não é particularidade da regressão linear - é o esqueleto de
   praticamente todo o aprendizado supervisionado moderno, redes neurais incluídas.
2. **O gradiente é a única informação necessária.** Não precisamos ver a superfície
   inteira, só a inclinação local. Foi assim que trocamos $120^9$ avaliações por
   algumas centenas de passos.
3. **A escala dos dados muda o comportamento do otimizador.** O mesmo código, com o
   mesmo $\alpha$, diverge com o peso em quilos e converge com o peso padronizado.
4. **Erro pequeno não é o mesmo que modelo adequado.** Só o gráfico de resíduos
   contou que faltava algo - e a correção veio de repensar a variável, não de trocar
   o algoritmo.

### Na próxima aula

- regressão **multivariada** com $\theta^T x$ e gradiente vetorizado;
- regressão **não-linear** por termos polinomiais - e o reencontro com o *overfitting*;
- a **equação normal** como alternativa ao gradiente, e quando cada uma vale a pena.

---
# 15. Exercícios

Os dois primeiros são exatamente os exercícios dos slides. Nos dois casos, **faça
primeiro à mão, no papel**, e use o código só para conferir - o objetivo é você sentir
a mecânica do algoritmo, e para isso a calculadora é melhor professora que o Python.

### Exercício 1 - Três iterações à mão *(exercício dos slides)*

> Considere os pontos $(0,3)$, $(5,4)$ e $(10,5)$. Encontre a reta por gradiente
> descendente, sem esquecer o coeficiente linear $\theta_0$. Faça **três iterações**,
> com $\alpha = 0{,}01$ e todos os pesos iguais a 1.

**(a)** Faça as três iterações no papel. Para cada uma, escreva $h(x^{(i)})$ para os
três pontos, os três erros, as duas derivadas e os novos $\theta_0$ e $\theta_1$.
Depois execute a célula e confira linha por linha.

**(b)** A solução exata deste problema é $\theta_0 = 3$ e $\theta_1 = 0{,}2$ (confira:
a reta passa exatamente pelos três pontos). Depois de três iterações, a que distância
você está? Quantas iterações seriam necessárias?

**(c)** Olhe a coluna de $J$. Ele diminuiu nas três iterações? E olhe $\theta_1$: ele
está indo na direção certa (de 1 para 0,2)? E $\theta_0$?

**(d)** A segunda célula repete o exercício com outros valores de $\alpha$. Qual
converge mais rápido? Algum diverge? Compare o limite que você encontrou aqui com o
limite de ~2 que vimos na seção 10, com o atributo padronizado - e explique a diferença
olhando os valores de $x$ dos dois problemas.

In [ ]:
# (a) - execute e compare com a sua conta no papel
x_e1 = np.array([0.0, 5.0, 10.0])
y_e1 = np.array([3.0, 4.0, 5.0])

t0, t1, alfa_e1 = 1.0, 1.0, 0.01
print(f"{'iter':>4} {'h(x1)':>8} {'h(x2)':>8} {'h(x3)':>8} "
      f"{'dJ/dt0':>9} {'dJ/dt1':>9} {'theta0':>9} {'theta1':>9} {'J':>9}")
print(f"{0:>4} {'':>8} {'':>8} {'':>8} {'':>9} {'':>9} {t0:>9.6f} {t1:>9.6f} "
      f"{custo(x_e1, y_e1, t0, t1):>9.6f}")
for k in range(1, 4):
    h = hipotese(x_e1, t0, t1)
    d0, d1 = gradiente(x_e1, y_e1, t0, t1)
    t0, t1 = t0 - alfa_e1 * d0, t1 - alfa_e1 * d1
    print(f"{k:>4} {h[0]:>8.4f} {h[1]:>8.4f} {h[2]:>8.4f} {d0:>9.4f} {d1:>9.4f} "
          f"{t0:>9.6f} {t1:>9.6f} {custo(x_e1, y_e1, t0, t1):>9.6f}")

print(f"\nSolução exata: theta0 = 3, theta1 = 0,2  (J = {custo(x_e1, y_e1, 3.0, 0.2):.1e})")

In [ ]:
# (d) - o mesmo exercício com vários alfas, 300 iterações
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for a, cor in zip([0.001, 0.01, 0.03, 0.045, 0.05], CORES):
    r = gradiente_descendente(x_e1, y_e1, 1.0, 1.0, alfa=a, n_iter=300)
    hj = r["historico_J"]
    ax.plot(hj, color=cor, linewidth=2, label=fr"$\alpha$={a}: $J$={hj[-1]:.3g}")
    print(f"alfa = {a:<6} → theta0 = {r['theta0']:12.4g}  theta1 = {r['theta1']:12.4g}  "
          f"J = {hj[-1]:.6g}")
ax.set_yscale("log")
ax.set_xlabel("iteração"); ax.set_ylabel("$J$ (escala log)")
ax.set_title("Exercício 1 (d) - efeito de $\\alpha$")
ax.legend(fontsize=8.5)
plt.tight_layout()
plt.show()

**Respostas de (a) a (d):** *(escreva aqui)*

### Exercício 2 - Cinco iterações com quatro variáveis *(exercício II dos slides)*

> Considere a base abaixo. Faça **cinco iterações** do gradiente descendente com
> $\alpha = 0{,}01$ e todos os pesos iguais a 1. Faça o gráfico da função de custo
> para mostrar a evolução do erro ao longo das iterações.

| Nota Disc. 1 | Nota Disc. 2 | Nota Disc. 3 | Nota Disc. 4 | **Nota IA** |
|---|---|---|---|---|
| 8,5 | 9,0 | 10,0 | 10,0 | **8,0** |
| 9,0 | 2,0 | 5,5 | 6,0 | **7,5** |
| 3,5 | 5,0 | 6,0 | 7,0 | **5,0** |
| 4,0 | 2,0 | 2,5 | 7,5 | **6,6** |

Aqui há **quatro** atributos, e a versão multivariada do algoritmo só será apresentada
na próxima aula - então **o código já vem pronto**. O laço é o mesmo de hoje; o que
muda é que $\theta$ passou a ser um vetor e a hipótese, um produto interno
$h(x) = \theta^T x$ (com $x_0 = 1$ para acomodar o intercepto).

**(a)** Execute e confira as cinco iterações à mão (pelo menos a primeira, inteira).

**(b)** O custo diminuiu nas cinco iterações? Olhe o gráfico: ele já convergiu?

**(c)** Compare os quatro pesos finais. Qual disciplina o modelo está considerando
mais importante para prever a nota de IA? Cuidado com a interpretação: depois de
apenas 5 iterações, esse valor diz mais sobre o **ponto de partida e as escalas** do
que sobre a realidade. Por quê?

**(d)** Mude `n_iter_e2` para 5000 e rode de novo. O modelo acerta as quatro notas -
**exatamente**, com erro da ordem de $10^{-11}$. Com 4 exemplos e 5 parâmetros, isso
deveria te deixar tranquilo ou desconfiado? Volte à discussão de *overfitting* e à
Navalha de Ockham da aula 01. O que você espera que aconteça com a nota prevista de um
quinto aluno?

In [ ]:
# Código pronto - a versão multivariada, que veremos na próxima aula.
notas = np.array([
    [8.5, 9.0, 10.0, 10.0],
    [9.0, 2.0,  5.5,  6.0],
    [3.5, 5.0,  6.0,  7.0],
    [4.0, 2.0,  2.5,  7.5],
])
nota_ia = np.array([8.0, 7.5, 5.0, 6.6])

# X com a coluna de 1s na frente (o x0 = 1 dos slides), para o theta0 entrar no vetor
X_e2 = np.hstack([np.ones((notas.shape[0], 1)), notas])
theta_e2 = np.ones(X_e2.shape[1])        # "todos os pesos iguais a 1"
alfa_e2, n_iter_e2 = 0.01, 5


def custo_multi(X, y, theta):
    erros = X @ theta - y
    return float(erros @ erros / (2 * y.size))


hist = [custo_multi(X_e2, nota_ia, theta_e2)]
print(f"iter  {'J':>10}   theta")
print(f"{0:>4}  {hist[0]:>10.6f}   {np.round(theta_e2, 4)}")
for k in range(1, n_iter_e2 + 1):
    grad = X_e2.T @ (X_e2 @ theta_e2 - nota_ia) / nota_ia.size   # todas as derivadas de uma vez
    theta_e2 = theta_e2 - alfa_e2 * grad                          # atualização simultânea
    hist.append(custo_multi(X_e2, nota_ia, theta_e2))
    print(f"{k:>4}  {hist[-1]:>10.6f}   {np.round(theta_e2, 4)}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(len(hist)), hist, marker="o", color=CORES[0], linewidth=2)
ax.set_xlabel("iteração"); ax.set_ylabel(r"$J(\theta)$")
ax.set_title("Exercício 2 - evolução do erro em 5 iterações")
ax.set_xticks(range(len(hist)))
plt.tight_layout()
plt.show()

print(f"\nPrevisões após {n_iter_e2} iterações: {np.round(X_e2 @ theta_e2, 3)}")
print(f"Notas reais:                     {nota_ia}")

**Respostas de (a) a (d):** *(escreva aqui)*

### Exercício 3 - Um atributo diferente

A célula abaixo repete todo o pipeline de hoje - padronizar, treinar por gradiente
descendente, desnormalizar, avaliar - para **cada** um dos atributos numéricos da base
de carros, um por vez.

**(a)** Qual atributo isolado prevê melhor o consumo? Qual prevê pior? Isso combina
com a sua intuição sobre carros?

**(b)** Todos os modelos usaram $\alpha = 0{,}1$ e 400 iterações, e todos convergiram.
Se tivéssemos usado os atributos **sem padronizar**, isso ainda seria verdade? Por quê?
(Compare as escalas na tabela de `describe()` da seção 2.)

**(c)** O sinal de $\theta_1$ é o esperado em cada caso? Explique o sinal do atributo
`ano`.

**(d)** O melhor $R^2$ isolado é ~0,7. Se juntássemos **todos** os atributos num único
modelo multivariado, você espera um $R^2$ maior, menor ou igual? Ele poderia ficar
menor? (Pense no que acontece com $J$ ao acrescentar um parâmetro.)

In [ ]:
atributos = ["peso_kg", "potencia", "cilindrada", "cilindros", "aceleracao", "ano"]
linhas = []

for atr in atributos:
    dados = carros[[atr, "consumo_kml"]].dropna()
    xi = dados[atr].to_numpy(dtype=float)
    yi = dados["consumo_kml"].to_numpy(dtype=float)
    xi_z, mu_i, sd_i = padroniza(xi)
    mod = gradiente_descendente(xi_z, yi, 0.0, 0.0, alfa=0.1, n_iter=400)
    b0, b1 = desnormaliza(mod["theta0"], mod["theta1"], mu_i, sd_i)
    linhas.append({
        "atributo": atr, "n": xi.size, "theta0": b0, "theta1": b1,
        "J": custo(xi, yi, b0, b1), "R2": r2_score(yi, hipotese(xi, b0, b1)),
    })

tabela = pd.DataFrame(linhas).sort_values("R2", ascending=False).set_index("atributo")
print(tabela.to_string(float_format=lambda v: f"{v:.4f}"))

fig, axes = plt.subplots(2, 3, figsize=(13, 6.4))
for ax, atr in zip(axes.ravel(), tabela.index):
    dados = carros[[atr, "consumo_kml"]].dropna()
    xi = dados[atr].to_numpy(dtype=float)
    yi = dados["consumo_kml"].to_numpy(dtype=float)
    b0, b1 = tabela.loc[atr, "theta0"], tabela.loc[atr, "theta1"]
    gi = np.linspace(xi.min(), xi.max(), 60)
    ax.scatter(xi, yi, s=12, c=CORES[0], alpha=0.4, edgecolor="none")
    ax.plot(gi, hipotese(gi, b0, b1), color=CORES[3], linewidth=2.2)
    ax.set_title(f"{atr}  -  $R^2$ = {tabela.loc[atr, 'R2']:.3f}", fontsize=9.5)
    ax.set_xlabel(atr)
axes[0, 0].set_ylabel("consumo (km/l)"); axes[1, 0].set_ylabel("consumo (km/l)")
plt.tight_layout()
plt.show()

**Respostas de (a) a (d):** *(escreva aqui)*

### Exercício 4 - Perguntas sobre o que você implementou

Sem código. Responda com o que viu hoje.

**(a)** Por que a função de custo eleva os erros ao quadrado em vez de somar os
valores absolutos? Cite uma vantagem do quadrado e uma desvantagem (pense em um carro
com um valor de consumo absurdo, digitado errado na planilha).

**(b)** O fator $\frac{1}{2}$ em $J$ não muda o ponto de mínimo. Ele muda alguma coisa
no **comportamento do gradiente descendente**? (Olhe as fórmulas das derivadas.)

**(c)** Na seção 8.1 vimos que a atualização sequencial "também funciona". Escreva, em
duas ou três frases, o que exatamente ela minimiza - e por que isso não é o que
pedimos.

**(d)** Se a superfície de $J$ tivesse dois vales de profundidades diferentes, o
gradiente descendente encontraria o mais fundo? De que dependeria? Por que esse
problema **não** existe em regressão linear com erro quadrático?

**(e)** Padronizamos o atributo `x`, mas não o alvo `y`. O algoritmo funcionou. Em que
situação padronizar `y` também seria útil? E o que você teria de fazer com as
previsões depois?

**Respostas de (a) a (e):** *(escreva aqui)*

---
## Referências

- NG, A. *Machine Learning* - notas de aula (Stanford CS229 / Coursera). Material que
  serviu de base aos slides desta aula.
- MITCHELL, T. *Machine Learning*. McGraw-Hill, 1997. (capítulo 4)
- JAMES, G. et al. *An Introduction to Statistical Learning*. Springer, 2013. (capítulo 3)
- HASTIE, T.; TIBSHIRANI, R.; FRIEDMAN, J. *The Elements of Statistical Learning*.
  2. ed. Springer, 2009. (capítulo 3)
- QUINLAN, R. Combining instance-based and model-based learning. In: *Proceedings of
  the Tenth International Conference on Machine Learning*, 1993. (base Auto MPG)
- DE COCK, D. Ames, Iowa: alternative to the Boston housing data as an end of semester
  regression project. *Journal of Statistics Education*, v. 19, n. 3, 2011.